# Projeto (Seção 6) — solução do instrutor

## Especificação

`manufactured_date >= 1980-01-01`; portas entre 2 e 4, múltiplo de 2; `id_` não anulável com uuid4 gerado por instância.
Enunciado em [`06_projeto`](06_projeto.ipynb).

In [1]:
from datetime import date
from enum import Enum
from pydantic import BaseModel, ConfigDict, Field, UUID4, field_serializer
from pydantic.alias_generators import to_camel


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"

In [2]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

In [3]:
from uuid import UUID

expected_serialized_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'France',
    'licensePlate': 'AAA-BBB'
}

In [4]:
data_no_id = {
    "manufacturer": "BMW",
    "seriesName": "M4",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

## Solução

In [5]:
from uuid import uuid4

class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=to_camel,
    )

    id_: UUID4 = Field(alias="id", default_factory=uuid4)
    manufacturer: str
    series_name: str
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = False
    manufactured_date: date = Field(validation_alias="completionDate", ge=date(1980, 1, 1))
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD",
        serialization_alias="baseMSRPUSD"
    )
    vin: str
    number_of_doors: int = Field(
        default=4,
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
    )
    registration_country: str | None = None
    license_plate: str | None = None

    @field_serializer("manufactured_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

In [6]:
car = Automobile.model_validate(data)
print(f"car: {car!r}")

car: Automobile(id_=UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'), manufacturer='BMW', series_name='M4', type_=<AutomobileType.convertible: 'Convertible'>, is_electric=False, manufactured_date=datetime.date(2023, 1, 1), base_msrp_usd=93300.0, vin='1234567890', number_of_doors=2, registration_country='France', license_plate='AAA-BBB')


In [7]:
assert car.model_dump(by_alias=True) == expected_serialized_by_alias

In [8]:
car1 = Automobile.model_validate(data_no_id)
print(f"car1: {car1!r}")

car1: Automobile(id_=UUID('6a99aa22-51f7-432e-b88c-1e63104f5b69'), manufacturer='BMW', series_name='M4', type_=<AutomobileType.convertible: 'Convertible'>, is_electric=False, manufactured_date=datetime.date(2023, 1, 1), base_msrp_usd=93300.0, vin='1234567890', number_of_doors=2, registration_country='France', license_plate='AAA-BBB')


In [9]:
car2 = Automobile.model_validate(data_no_id)
print(f"car2: {car2!r}")

car2: Automobile(id_=UUID('defcf4a9-5763-4156-ac8c-62cdffed3af3'), manufacturer='BMW', series_name='M4', type_=<AutomobileType.convertible: 'Convertible'>, is_electric=False, manufactured_date=datetime.date(2023, 1, 1), base_msrp_usd=93300.0, vin='1234567890', number_of_doors=2, registration_country='France', license_plate='AAA-BBB')


In [10]:
assert car1.id_ is not None
assert car1.id_ != car2.id_

E conferindo que a validação das portas funciona como esperado:

In [11]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "1960-01-01",
    "msrpUSD": 93_300,
    "vin": "1234567890",
    "doors": 3,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

In [12]:
from pydantic import ValidationError


try:
    Automobile.model_validate(data)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Automobile
completionDate
  Input should be greater than or equal to 1980-01-01 [type=greater_than_equal, input_value='1960-01-01', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal
doors
  Input should be a multiple of 2 [type=multiple_of, input_value=3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/multiple_of
